# 02 · Baseline CNN  *(Onkar - part 1)*

**Goal:** build and train a CNN **from scratch** that every other model is compared against.

**Design** (see `fruitfresh/models.py`): 4 blocks of *Conv-BatchNorm-ReLU ×2 + MaxPool* (32→64→128→256 filters) → global average pooling → dropout 0.3 → 6-way softmax. About 1.2 M parameters.
Training: Adam (lr 1e-3), categorical cross-entropy, class weights, random augmentation, early stopping on validation loss.

**Only the validation split is used here.** The test split is never loaded.

In [ ]:
# ============ SETUP - run this first after every fresh Colab runtime ============
import os, sys, subprocess
from pathlib import Path

REPO_URL = ""   # e.g. "https://github.com/<your-username>/fruit-freshness-classifier.git"
                # leave "" if you uploaded/unzipped the project folder yourself

IN_COLAB = "google.colab" in sys.modules
PROJECT_DIR = Path("/content/fruit-freshness-classifier") if IN_COLAB else Path.cwd().parent
if not (PROJECT_DIR / "src").exists():
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    else:
        raise SystemExit(f"Project folder not found at {PROJECT_DIR}. Set REPO_URL above or upload the project there.")
elif REPO_URL and IN_COLAB:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

if os.getenv("FRUIT_SKIP_PIP") != "1":     # only kagglehub is needed; Colab already has TensorFlow etc.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-colab.txt")], check=True)
sys.path.insert(0, str(PROJECT_DIR / "src"))

import json
import matplotlib.pyplot as plt
import pandas as pd
from fruitfresh import config
from fruitfresh.colab import setup_workspace, gpu_summary

WORK_DIR = setup_workspace()      # Colab: Google Drive (survives runtime resets)
SEED = config.SEED
print(gpu_summary())
print("Project folder :", PROJECT_DIR)
print("Work folder    :", WORK_DIR, "(models, histories, results are saved here)")

In [ ]:
from fruitfresh.data_index import get_dataset_root
from fruitfresh.splits import load_split

DATASET_ROOT = get_dataset_root()
SPLIT_DIR = WORK_DIR / "data" / "splits"
if not SPLIT_DIR.exists():
    SPLIT_DIR = PROJECT_DIR / "data" / "splits"      # the committed copy of the team split
train_df, val_df, _ = load_split(SPLIT_DIR)           # the TEST split is deliberately ignored (`_`)
print(f"train {len(train_df)} images / {train_df.source_id.nunique()} photos | "
      f"validation {len(val_df)} images / {val_df.source_id.nunique()} photos")
print("Split folder:", SPLIT_DIR)

## 1 · Look at the model

In [ ]:
from fruitfresh.models import build_model
model = build_model("baseline_cnn", pretrained=False)
model.summary()

## 2 · Sanity check: can the model memorise a tiny batch?
A correct pipeline must reach ~100 % training accuracy on 64 images. If it cannot, something is broken (labels, pre-processing, learning rate) - fix this **before** spending hours on training.

In [ ]:
import tensorflow as tf
from fruitfresh.datasets import make_dataset
tiny = train_df.sample(64, random_state=0)
tiny_ds = make_dataset(tiny, DATASET_ROOT, batch_size=16, training=False, augment=False)
check = build_model("baseline_cnn", pretrained=False)
check.compile(tf.keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
h = check.fit(tiny_ds, epochs=25, verbose=0)
print("accuracy on the 64 training images after 25 epochs:", round(h.history["accuracy"][-1], 3))
assert h.history["accuracy"][-1] > 0.9, "Cannot overfit a tiny batch -> check labels / pre-processing"
del check

## 3 · Train
`force=False` means: if a trained model for this run already exists in Drive, it is **loaded instead of retrained** (safe after a Colab disconnect). Set `FORCE = True` to retrain.

In [ ]:
from fruitfresh.train import train_model, run_dir_for
FORCE = False
info = train_model("baseline_cnn", train_df, val_df, DATASET_ROOT, WORK_DIR, seed=SEED, force=FORCE, pretrained=False)
pd.Series({k: info[k] for k in ("epochs_run", "best_epoch", "best_val_loss", "best_val_accuracy", "params", "training_seconds")})

In [ ]:
from fruitfresh.viz import plot_history
RUN_DIR = run_dir_for(WORK_DIR, "baseline_cnn", SEED)
fig = plot_history(RUN_DIR / "history.csv", "Baseline CNN"); plt.show()

**How to read the curves**
* train ≫ validation accuracy and validation loss rising → **overfitting** (more augmentation / dropout / earlier stop).
* both low and flat → **underfitting** (longer training, larger model, higher learning rate).
* validation curve jumping around → small validation set / learning rate too high; trust the *trend*, not single epochs.
* the saved model is the epoch with the **lowest validation loss**, not the last epoch.

## 4 · Validation diagnostics

In [ ]:
from fruitfresh.evaluate import evaluate_validation, plot_confusion_matrix
m = evaluate_validation(RUN_DIR, val_df, DATASET_ROOT)
print(f"validation: accuracy {m['accuracy']:.4f} | macro-F1 {m['macro_f1']:.4f} ({m['n_images']} images, {m['n_sources']} photos)")
display(pd.DataFrame(m["per_class"]).T.round(3))
fig = plot_confusion_matrix(m["confusion_matrix"], title="Baseline CNN - validation"); plt.show()

Which classes are weakest? Note them (e.g. rotten vs fresh of the *same* fruit): the same classes usually stay hard for MobileNetV2, and Krishna's CNN analysis can start from here.

## Checklist before notebook 03
- [ ] the overfit-a-tiny-batch check passed
- [ ] `runs/baseline_cnn/seed42/model.keras`, `history.csv`, `train_info.json` exist in Drive
- [ ] you can explain the curves in 2 sentences (over/under-fitting?)